# LSTM, Transformer and ARIMA exploration

Historical comparison and equal-weight ensemble. The models use different validation horizons; see docs/EXPERIMENT_NOTES.md before interpreting the comparison.

Model logic is preserved from the course project. Paths are relative to this repository; generated files go to `outputs/ensemble/`. Saved outputs were cleared for publication.


In [ ]:
from pathlib import Path

# Locate the repository when launched from its root or notebooks directory.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data").is_dir() and (p / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch Jupyter from the repository root.")
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "ensemble"
for name in ("train.csv", "val.csv", "test.csv"):
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(f"Missing {DATA_DIR / name}. See data/README.md.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# 统计模型
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# 深度学习库
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import math

# 设置随机种子
np.random.seed(42)
torch.manual_seed(42)

# 设置绘图风格
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# ============================================================================
# 1. 数据加载和预处理
# ============================================================================

# def load_data(file_path):
#     """加载数据并添加周序列标签"""
#     try:
#         df = pd.read_csv(file_path, sep='\t')
#     except:
#         df = pd.read_csv(file_path)
    
#     # 添加周序列索引
#     df['Week'] = range(len(df))
    
#     # 重命名列以确保一致性
#     if 'Predict Target' in df.columns:
#         df.rename(columns={'Predict Target': 'Target'}, inplace=True)
    
#     return df

def load_data(file_path):
    """加载数据并添加周序列标签"""
    # 尝试不同的分隔符读取
    try:
        df = pd.read_csv(file_path)
    except:
        try:
            df = pd.read_csv(file_path, sep='\t')
        except:
            df = pd.read_csv(file_path, delimiter='\t')
    
    # 打印原始列名以便调试
    print(f"读取文件 {file_path}，列名: {df.columns.tolist()}")
    
    # 清理列名（去除空格）
    df.columns = df.columns.str.strip()
    
    # 添加周序列索引
    df['Week'] = range(len(df))
    
    # 统一列名处理
    # 检查可能的目标列名称
    target_columns = ['Predict Target', 'Target', 'predict target', 'target']
    date_columns = ['Relative Date', 'Date', 'relative date', 'date']
    
    # 查找目标列
    target_col = None
    for col in target_columns:
        if col in df.columns:
            target_col = col
            break
    
    # 查找日期列
    date_col = None
    for col in date_columns:
        if col in df.columns:
            date_col = col
            break
    
    # 重命名为标准列名
    if target_col and target_col != 'Target':
        df.rename(columns={target_col: 'Target'}, inplace=True)
    
    if date_col and date_col != 'Relative Date':
        df.rename(columns={date_col: 'Relative Date'}, inplace=True)
    
    # 验证是否成功找到目标列
    if 'Target' not in df.columns:
        print(f"警告: 未找到目标列。可用列: {df.columns.tolist()}")
        # 如果第二列是数值型，假定它是目标列
        if len(df.columns) >= 2 and pd.api.types.is_numeric_dtype(df.iloc[:, 1]):
            df.rename(columns={df.columns[1]: 'Target'}, inplace=True)
            print(f"已将第二列 '{df.columns[1]}' 重命名为 'Target'")
    
    # 确保Target列是数值型
    if 'Target' in df.columns:
        df['Target'] = pd.to_numeric(df['Target'], errors='coerce')
    
    print(f"处理后的列名: {df.columns.tolist()}")
    print(f"数据预览:\n{df.head()}\n")
    
    return df

# ============================================================================
# 2. 数据集类定义
# ============================================================================

class TimeSeriesDataset(Dataset):
    """时间序列数据集"""
    def __init__(self, sequences, targets):
        self.sequences = torch.FloatTensor(sequences)
        self.targets = torch.FloatTensor(targets)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

def create_sequences(data, seq_length=12):
    """创建序列数据"""
    sequences = []
    targets = []
    
    for i in range(len(data) - seq_length):
        seq = data[i:i+seq_length]
        tar = data[i+seq_length]
        sequences.append(seq)
        targets.append(tar)
    
    return np.array(sequences), np.array(targets)

# ============================================================================
# 3. LSTM模型定义
# ============================================================================

class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=2, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        
        self.fc1 = nn.Linear(hidden_size, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]
        out = self.dropout(self.relu(self.fc1(out)))
        out = self.dropout(self.relu(self.fc2(out)))
        out = self.fc3(out)
        return out

# ============================================================================
# 4. Transformer模型定义
# ============================================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class TransformerModel(nn.Module):
    def __init__(self, input_size=1, d_model=64, nhead=4, num_layers=2, dropout=0.2):
        super(TransformerModel, self).__init__()
        self.d_model = d_model
        
        # 输入投影
        self.input_projection = nn.Linear(input_size, d_model)
        
        # 位置编码
        self.pos_encoder = PositionalEncoding(d_model)
        
        # Transformer编码器
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=256,
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        
        # 输出层
        self.fc1 = nn.Linear(d_model, 32)
        self.fc2 = nn.Linear(32, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # x shape: [batch_size, seq_len, input_size]
        x = self.input_projection(x) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        x = x[:, -1, :]  # 取最后一个时间步
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

# ============================================================================
# 5. ARIMA模型类
# ============================================================================

class ARIMAModel:
    def __init__(self, order=(1, 1, 1)):
        self.order = order
        self.model = None
        self.fitted_model = None
        
    def fit(self, train_data):
        """训练ARIMA模型"""
        self.model = ARIMA(train_data, order=self.order)
        self.fitted_model = self.model.fit()
        return self.fitted_model
    
    def predict(self, steps):
        """预测未来steps步"""
        if self.fitted_model is None:
            raise ValueError("Model not fitted yet!")
        forecast = self.fitted_model.forecast(steps=steps)
        return forecast
    
    def get_params(self):
        """获取模型参数"""
        if self.fitted_model is None:
            return None
        return self.fitted_model.params

# ============================================================================
# 6. 训练函数
# ============================================================================

def train_deep_model(model, train_loader, val_loader, model_name, epochs=100, lr=0.001, patience=15):
    """训练深度学习模型（LSTM或Transformer）"""
    print(f"\n{'='*60}")
    print(f"训练 {model_name} 模型")
    print(f"{'='*60}")
    
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5 #, verbose=False
    )
    
    best_val_loss = float('inf')
    patience_counter = 0
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        # 训练阶段
        model.train()
        train_loss = 0
        for sequences, targets in train_loader:
            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs.squeeze(), targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        
        # 验证阶段
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for sequences, targets in val_loader:
                outputs = model(sequences)
                loss = criterion(outputs.squeeze(), targets)
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        
        scheduler.step(val_loss)
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')
        
        # 早停
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), str(OUTPUT_DIR / f'best_{model_name.lower()}_model.pth'))
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break
    
    # 加载最佳模型
    model.load_state_dict(torch.load(str(OUTPUT_DIR / f'best_{model_name.lower()}_model.pth')))
    
    return model, train_losses, val_losses

def train_arima_model(train_data, order=(5, 1, 2)):
    """训练ARIMA模型"""
    print(f"\n{'='*60}")
    print(f"训练 ARIMA{order} 模型")
    print(f"{'='*60}")
    
    arima_model = ARIMAModel(order=order)
    fitted_model = arima_model.fit(train_data)
    
    print("\nARIMA模型摘要:")
    print(fitted_model.summary())
    
    return arima_model

# ============================================================================
# 7. 评估指标
# ============================================================================

def calculate_metrics(y_true, y_pred):
    """计算各种评估指标"""
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    
    # 避免除零错误
    mask = y_true != 0
    if np.sum(mask) > 0:
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = 0
    
    return {
        'MSE': mse,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE': mape
    }

# ============================================================================
# 8. 可视化函数
# ============================================================================

def plot_training_history(train_losses, val_losses, model_name, save_path):
    """绘制训练历史"""
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(train_losses, label='Train Loss', linewidth=2)
    plt.plot(val_losses, label='Val Loss', linewidth=2)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title(f'{model_name} - Training History', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.plot(train_losses, label='Train Loss', linewidth=2)
    plt.plot(val_losses, label='Val Loss', linewidth=2)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss (log scale)', fontsize=12)
    plt.title(f'{model_name} - Training History (Log Scale)', fontsize=14, fontweight='bold')
    plt.yscale('log')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"训练历史图已保存至 {save_path}")

def plot_predictions(y_true, y_pred, model_name, save_path):
    """绘制预测结果"""
    plt.figure(figsize=(15, 5))
    
    # 时间序列对比
    plt.subplot(1, 2, 1)
    plt.plot(y_true, label='Actual', marker='o', linewidth=2, markersize=4)
    plt.plot(y_pred, label='Predicted', marker='x', linewidth=2, markersize=4)
    plt.xlabel('Week', fontsize=12)
    plt.ylabel('Target Value', fontsize=12)
    plt.title(f'{model_name} - Predictions vs Actual', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    
    # 散点图
    plt.subplot(1, 2, 2)
    plt.scatter(y_true, y_pred, alpha=0.6, s=50)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    plt.xlabel('Actual Value', fontsize=12)
    plt.ylabel('Predicted Value', fontsize=12)
    plt.title(f'{model_name} - Scatter Plot', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"预测结果图已保存至 {save_path}")

def plot_residuals(y_true, y_pred, model_name, save_path):
    """绘制残差分析"""
    residuals = y_true - y_pred
    
    plt.figure(figsize=(15, 10))
    
    # 残差时间序列
    plt.subplot(2, 2, 1)
    plt.plot(residuals, marker='o', linewidth=2, markersize=4)
    plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
    plt.xlabel('Week', fontsize=12)
    plt.ylabel('Residual', fontsize=12)
    plt.title(f'{model_name} - Residuals Over Time', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    
    # 残差直方图
    plt.subplot(2, 2, 2)
    plt.hist(residuals, bins=30, edgecolor='black', alpha=0.7)
    plt.xlabel('Residual', fontsize=12)
    plt.ylabel('Frequency', fontsize=12)
    plt.title(f'{model_name} - Residual Distribution', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    
    # Q-Q图
    plt.subplot(2, 2, 3)
    from scipy import stats
    stats.probplot(residuals, dist="norm", plot=plt)
    plt.title(f'{model_name} - Q-Q Plot', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    
    # 残差vs预测值
    plt.subplot(2, 2, 4)
    plt.scatter(y_pred, residuals, alpha=0.6, s=50)
    plt.axhline(y=0, color='r', linestyle='--', linewidth=2)
    plt.xlabel('Predicted Value', fontsize=12)
    plt.ylabel('Residual', fontsize=12)
    plt.title(f'{model_name} - Residuals vs Predictions', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"残差分析图已保存至 {save_path}")

def plot_model_comparison(results_dict, train_df, val_df, test_df, test_predictions_dict, save_path):
    """绘制模型对比图"""
    fig = plt.figure(figsize=(20, 12))
    
    # 1. 完整时间序列对比
    ax1 = plt.subplot(3, 2, 1)
    train_range = range(len(train_df))
    val_range = range(len(train_df), len(train_df) + len(val_df))
    test_range = range(len(train_df) + len(val_df), len(train_df) + len(val_df) + len(test_df))
    
    ax1.plot(train_range, train_df['Target'], label='Train', linewidth=2, alpha=0.7)
    ax1.plot(val_range, val_df['Target'], label='Val', linewidth=2, alpha=0.7)
    
    colors = ['red', 'green', 'purple']
    for idx, (model_name, predictions) in enumerate(test_predictions_dict.items()):
        ax1.plot(test_range, predictions, label=f'{model_name} Predictions', 
                linewidth=2, linestyle='--', color=colors[idx], marker='o', markersize=4)
    
    ax1.set_xlabel('Week', fontsize=12)
    ax1.set_ylabel('Target Value', fontsize=12)
    ax1.set_title('Complete Time Series - All Models', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # 2. 验证集性能对比（时间序列）
    ax2 = plt.subplot(3, 2, 2)
    ax2.plot(val_df['Target'].values, label='Actual', linewidth=2.5, color='black', marker='o', markersize=5)
    
    colors = ['red', 'green', 'purple']
    for idx, (model_name, result) in enumerate(results_dict.items()):
        if 'val_predictions' in result:
            ax2.plot(result['val_predictions'], label=model_name, 
                    linewidth=2, linestyle='--', color=colors[idx], marker='x', markersize=4)
    
    ax2.set_xlabel('Week', fontsize=12)
    ax2.set_ylabel('Target Value', fontsize=12)
    ax2.set_title('Validation Set - Model Comparison', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    # 3. 指标对比 - MSE
    ax3 = plt.subplot(3, 2, 3)
    models = list(results_dict.keys())
    mse_values = [results_dict[m]['metrics']['MSE'] for m in models]
    bars = ax3.bar(models, mse_values, color=colors[:len(models)], alpha=0.7, edgecolor='black')
    ax3.set_ylabel('MSE', fontsize=12)
    ax3.set_title('Mean Squared Error Comparison', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 添加数值标签
    for bar in bars:
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=10)
    
    # 4. 指标对比 - RMSE
    ax4 = plt.subplot(3, 2, 4)
    rmse_values = [results_dict[m]['metrics']['RMSE'] for m in models]
    bars = ax4.bar(models, rmse_values, color=colors[:len(models)], alpha=0.7, edgecolor='black')
    ax4.set_ylabel('RMSE', fontsize=12)
    ax4.set_title('Root Mean Squared Error Comparison', fontsize=14, fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='y')
    
    for bar in bars:
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=10)
    
    # 5. 指标对比 - MAE
    ax5 = plt.subplot(3, 2, 5)
    mae_values = [results_dict[m]['metrics']['MAE'] for m in models]
    bars = ax5.bar(models, mae_values, color=colors[:len(models)], alpha=0.7, edgecolor='black')
    ax5.set_ylabel('MAE', fontsize=12)
    ax5.set_title('Mean Absolute Error Comparison', fontsize=14, fontweight='bold')
    ax5.grid(True, alpha=0.3, axis='y')
    
    for bar in bars:
        height = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=10)
    
    # 6. 指标对比 - MAPE
    ax6 = plt.subplot(3, 2, 6)
    mape_values = [results_dict[m]['metrics']['MAPE'] for m in models]
    bars = ax6.bar(models, mape_values, color=colors[:len(models)], alpha=0.7, edgecolor='black')
    ax6.set_ylabel('MAPE (%)', fontsize=12)
    ax6.set_title('Mean Absolute Percentage Error Comparison', fontsize=14, fontweight='bold')
    ax6.grid(True, alpha=0.3, axis='y')
    
    for bar in bars:
        height = bar.get_height()
        ax6.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}%', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"\n模型对比图已保存至 {save_path}")

def plot_arima_diagnostics(train_data, arima_model, save_path):
    """绘制ARIMA诊断图"""
    fig = plt.figure(figsize=(15, 10))
    
    # 1. ACF图
    ax1 = plt.subplot(2, 2, 1)
    plot_acf(train_data, lags=40, ax=ax1)
    ax1.set_title('Autocorrelation Function (ACF)', fontsize=12, fontweight='bold')
    
    # 2. PACF图
    ax2 = plt.subplot(2, 2, 2)
    plot_pacf(train_data, lags=40, ax=ax2)
    ax2.set_title('Partial Autocorrelation Function (PACF)', fontsize=12, fontweight='bold')
    
    # 3. 残差分析
    if arima_model.fitted_model is not None:
        residuals = arima_model.fitted_model.resid
        
        ax3 = plt.subplot(2, 2, 3)
        ax3.plot(residuals, linewidth=2)
        ax3.axhline(y=0, color='r', linestyle='--', linewidth=2)
        ax3.set_xlabel('Week', fontsize=12)
        ax3.set_ylabel('Residual', fontsize=12)
        ax3.set_title('ARIMA Residuals', fontsize=12, fontweight='bold')
        ax3.grid(True, alpha=0.3)
        
        ax4 = plt.subplot(2, 2, 4)
        ax4.hist(residuals, bins=30, edgecolor='black', alpha=0.7)
        ax4.set_xlabel('Residual', fontsize=12)
        ax4.set_ylabel('Frequency', fontsize=12)
        ax4.set_title('ARIMA Residual Distribution', fontsize=12, fontweight='bold')
        ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"ARIMA诊断图已保存至 {save_path}")

# ============================================================================
# 9. 主函数
# ============================================================================

def main():
    print("="*80)
    print("多模型时间序列预测系统 (LSTM + ARIMA + Transformer)")
    print("="*80)
    
    # ========== 1. 加载数据 ==========
    print("\n" + "="*80)
    print("1. 加载数据")
    print("="*80)
    
    train_df = load_data(str(DATA_DIR / 'train.csv'))
    val_df = load_data(str(DATA_DIR / 'val.csv'))
    test_df = load_data(str(DATA_DIR / 'test.csv'))
    
    print(f"训练集大小: {len(train_df)}")
    print(f"验证集大小: {len(val_df)}")
    print(f"测试集大小: {len(test_df)}")
    print(f"\n数据统计:")
    print(f"训练集 - Mean: {train_df['Target'].mean():.2f}, Std: {train_df['Target'].std():.2f}")
    print(f"验证集 - Mean: {val_df['Target'].mean():.2f}, Std: {val_df['Target'].std():.2f}")
    
    # 合并训练和验证数据用于最终预测
    full_train_df = pd.concat([train_df, val_df], ignore_index=True)
    full_train_df['Week'] = range(len(full_train_df))
    
    # 设置序列长度
    seq_length = 12
    
    # ========== 2. 准备深度学习模型数据 ==========
    print("\n" + "="*80)
    print("2. 准备深度学习模型数据")
    print("="*80)
    
    # 准备数据
    train_data = train_df['Target'].values.reshape(-1, 1)
    val_data = val_df['Target'].values.reshape(-1, 1)
    
    # 标准化
    scaler = StandardScaler()
    train_data_scaled = scaler.fit_transform(train_data)
    val_data_scaled = scaler.transform(val_data)
    
    # 创建序列
    X_train, y_train = create_sequences(train_data_scaled, seq_length)
    X_val, y_val = create_sequences(val_data_scaled, seq_length)
    
    print(f"训练序列形状: {X_train.shape}")
    print(f"验证序列形状: {X_val.shape}")
    
    # 创建数据加载器
    train_dataset = TimeSeriesDataset(X_train, y_train)
    val_dataset = TimeSeriesDataset(X_val, y_val)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    # 用于存储所有模型的结果
    results = {}
    
    # ========== 3. 训练LSTM模型 ==========
    print("\n" + "="*80)
    print("3. 训练LSTM模型")
    print("="*80)
    
    lstm_model = LSTMModel(input_size=1, hidden_size=128, num_layers=2, dropout=0.2)
    lstm_model, lstm_train_losses, lstm_val_losses = train_deep_model(
        lstm_model, train_loader, val_loader, "LSTM",
        epochs=100, lr=0.001, patience=15
    )
    
    # 绘制LSTM训练历史
    plot_training_history(lstm_train_losses, lstm_val_losses, "LSTM", str(OUTPUT_DIR / 'lstm_training_history.png'))
    
    # LSTM验证集评估
    lstm_model.eval()
    lstm_val_predictions = []
    with torch.no_grad():
        for sequences, _ in val_loader:
            outputs = lstm_model(sequences)
            lstm_val_predictions.extend(outputs.squeeze().numpy())
    
    lstm_val_predictions = np.array(lstm_val_predictions).reshape(-1, 1)
    lstm_val_predictions_rescaled = scaler.inverse_transform(lstm_val_predictions).flatten()
    lstm_val_actual = val_df['Target'].values[seq_length:]
    
    lstm_metrics = calculate_metrics(lstm_val_actual, lstm_val_predictions_rescaled)
    
    # 绘制LSTM预测结果
    plot_predictions(lstm_val_actual, lstm_val_predictions_rescaled, "LSTM", str(OUTPUT_DIR / 'lstm_predictions.png'))
    plot_residuals(lstm_val_actual, lstm_val_predictions_rescaled, "LSTM", str(OUTPUT_DIR / 'lstm_residuals.png'))
    
    results['LSTM'] = {
        'model': lstm_model,
        'metrics': lstm_metrics,
        'val_predictions': lstm_val_predictions_rescaled,
        'train_losses': lstm_train_losses,
        'val_losses': lstm_val_losses
    }
    
    print("\nLSTM验证集性能指标:")
    print("-" * 40)
    for metric, value in lstm_metrics.items():
        print(f"{metric}: {value:.4f}")
    
    # ========== 4. 训练Transformer模型 ==========
    print("\n" + "="*80)
    print("4. 训练Transformer模型")
    print("="*80)
    
    transformer_model = TransformerModel(input_size=1, d_model=64, nhead=4, num_layers=2, dropout=0.2)
    transformer_model, transformer_train_losses, transformer_val_losses = train_deep_model(
        transformer_model, train_loader, val_loader, "Transformer",
        epochs=100, lr=0.001, patience=15
    )
    
    # 绘制Transformer训练历史
    plot_training_history(transformer_train_losses, transformer_val_losses, "Transformer", str(OUTPUT_DIR / 'transformer_training_history.png'))
    
    # Transformer验证集评估
    transformer_model.eval()
    transformer_val_predictions = []
    with torch.no_grad():
        for sequences, _ in val_loader:
            outputs = transformer_model(sequences)
            transformer_val_predictions.extend(outputs.squeeze().numpy())
    
    transformer_val_predictions = np.array(transformer_val_predictions).reshape(-1, 1)
    transformer_val_predictions_rescaled = scaler.inverse_transform(transformer_val_predictions).flatten()
    transformer_val_actual = val_df['Target'].values[seq_length:]
    
    transformer_metrics = calculate_metrics(transformer_val_actual, transformer_val_predictions_rescaled)
    
    # 绘制Transformer预测结果
    plot_predictions(transformer_val_actual, transformer_val_predictions_rescaled, "Transformer", str(OUTPUT_DIR / 'transformer_predictions.png'))
    plot_residuals(transformer_val_actual, transformer_val_predictions_rescaled, "Transformer", str(OUTPUT_DIR / 'transformer_residuals.png'))
    
    results['Transformer'] = {
        'model': transformer_model,
        'metrics': transformer_metrics,
        'val_predictions': transformer_val_predictions_rescaled,
        'train_losses': transformer_train_losses,
        'val_losses': transformer_val_losses
    }
    
    print("\nTransformer验证集性能指标:")
    print("-" * 40)
    for metric, value in transformer_metrics.items():
        print(f"{metric}: {value:.4f}")
    
    # ========== 5. 训练ARIMA模型 ==========
    print("\n" + "="*80)
    print("5. 训练ARIMA模型")
    print("="*80)
    
    # 绘制ARIMA诊断图
    plot_arima_diagnostics(train_df['Target'].values, ARIMAModel(), str(OUTPUT_DIR / 'arima_diagnostics_initial.png'))
    
    # 训练ARIMA模型
    arima_train_data = train_df['Target'].values
    arima_model = train_arima_model(arima_train_data, order=(5, 1, 2))
    
    # ARIMA验证集评估
    arima_val_predictions = arima_model.predict(steps=len(val_df))
    arima_val_actual = val_df['Target'].values
    
    arima_metrics = calculate_metrics(arima_val_actual, arima_val_predictions)
    
    # 绘制ARIMA预测结果
    plot_predictions(arima_val_actual, arima_val_predictions, "ARIMA", str(OUTPUT_DIR / 'arima_predictions.png'))
    plot_residuals(arima_val_actual, arima_val_predictions, "ARIMA", str(OUTPUT_DIR / 'arima_residuals.png'))
    plot_arima_diagnostics(arima_train_data, arima_model, str(OUTPUT_DIR / 'arima_diagnostics_fitted.png'))
    
    results['ARIMA'] = {
        'model': arima_model,
        'metrics': arima_metrics,
        'val_predictions': arima_val_predictions
    }
    
    print("\nARIMA验证集性能指标:")
    print("-" * 40)
    for metric, value in arima_metrics.items():
        print(f"{metric}: {value:.4f}")
    
    # ========== 6. 模型对比分析 ==========
    print("\n" + "="*80)
    print("6. 模型对比分析")
    print("="*80)
    
    print("\n所有模型性能对比:")
    print("="*80)
    
    # 创建对比表格
    comparison_df = pd.DataFrame({
        model_name: result['metrics'] 
        for model_name, result in results.items()
    }).T
    
    print(comparison_df.to_string())
    
    # 保存对比结果
    comparison_df.to_csv(str(OUTPUT_DIR / 'model_comparison.csv'))
    print("\n对比结果已保存至 model_comparison.csv")
    
    # 找出最佳模型
    best_model_by_metric = {}
    for metric in ['MSE', 'RMSE', 'MAE', 'MAPE']:
        best_model = comparison_df[metric].idxmin()
        best_value = comparison_df[metric].min()
        best_model_by_metric[metric] = (best_model, best_value)
        print(f"\n最佳{metric}模型: {best_model} ({best_value:.4f})")
    
    # ========== 7. 使用完整数据重新训练并生成测试集预测 ==========
    print("\n" + "="*80)
    print("7. 使用完整数据重新训练并生成测试集预测")
    print("="*80)
    
    test_predictions_dict = {}
    
    # 7.1 LSTM测试集预测
    print("\n7.1 LSTM测试集预测...")
    full_train_data = full_train_df['Target'].values.reshape(-1, 1)
    full_scaler = StandardScaler()
    full_train_data_scaled = full_scaler.fit_transform(full_train_data)
    
    X_full, y_full = create_sequences(full_train_data_scaled, seq_length)
    full_dataset = TimeSeriesDataset(X_full, y_full)
    full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True)
    
    final_lstm_model = LSTMModel(input_size=1, hidden_size=128, num_layers=2, dropout=0.2)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(final_lstm_model.parameters(), lr=0.001)
    
    for epoch in range(50):
        final_lstm_model.train()
        for sequences, targets in full_loader:
            optimizer.zero_grad()
            outputs = final_lstm_model(sequences)
            loss = criterion(outputs.squeeze(), targets)
            loss.backward()
            optimizer.step()
    
    final_lstm_model.eval()
    lstm_test_predictions = []
    current_sequence = full_train_data_scaled[-seq_length:].reshape(1, seq_length, 1)
    
    for i in range(len(test_df)):
        with torch.no_grad():
            current_seq_tensor = torch.FloatTensor(current_sequence)
            pred = final_lstm_model(current_seq_tensor)
            pred_value = pred.item()
            lstm_test_predictions.append(pred_value)
            current_sequence = np.append(current_sequence[0, 1:, :], [[pred_value]], axis=0).reshape(1, seq_length, 1)
    
    lstm_test_predictions = np.array(lstm_test_predictions).reshape(-1, 1)
    lstm_test_predictions_rescaled = full_scaler.inverse_transform(lstm_test_predictions).flatten()
    test_predictions_dict['LSTM'] = lstm_test_predictions_rescaled
    
    # 7.2 Transformer测试集预测
    print("7.2 Transformer测试集预测...")
    final_transformer_model = TransformerModel(input_size=1, d_model=64, nhead=4, num_layers=2, dropout=0.2)
    optimizer = torch.optim.Adam(final_transformer_model.parameters(), lr=0.001)
    
    for epoch in range(50):
        final_transformer_model.train()
        for sequences, targets in full_loader:
            optimizer.zero_grad()
            outputs = final_transformer_model(sequences)
            loss = criterion(outputs.squeeze(), targets)
            loss.backward()
            optimizer.step()
    
    final_transformer_model.eval()
    transformer_test_predictions = []
    current_sequence = full_train_data_scaled[-seq_length:].reshape(1, seq_length, 1)
    
    for i in range(len(test_df)):
        with torch.no_grad():
            current_seq_tensor = torch.FloatTensor(current_sequence)
            pred = final_transformer_model(current_seq_tensor)
            pred_value = pred.item()
            transformer_test_predictions.append(pred_value)
            current_sequence = np.append(current_sequence[0, 1:, :], [[pred_value]], axis=0).reshape(1, seq_length, 1)
    
    transformer_test_predictions = np.array(transformer_test_predictions).reshape(-1, 1)
    transformer_test_predictions_rescaled = full_scaler.inverse_transform(transformer_test_predictions).flatten()
    test_predictions_dict['Transformer'] = transformer_test_predictions_rescaled
    
    # 7.3 ARIMA测试集预测
    print("7.3 ARIMA测试集预测...")
    final_arima_model = train_arima_model(full_train_df['Target'].values, order=(5, 1, 2))
    arima_test_predictions = final_arima_model.predict(steps=len(test_df))
    test_predictions_dict['ARIMA'] = arima_test_predictions
    
    # ========== 8. 生成综合对比图 ==========
    print("\n" + "="*80)
    print("8. 生成综合对比可视化")
    print("="*80)
    
    plot_model_comparison(results, train_df, val_df, test_df, test_predictions_dict, str(OUTPUT_DIR / 'comprehensive_model_comparison.png'))
    
    # ========== 9. 保存预测结果 ==========
    print("\n" + "="*80)
    print("9. 保存预测结果")
    print("="*80)
    
    # 保存各模型的测试集预测
    for model_name, predictions in test_predictions_dict.items():
        submission = pd.DataFrame({
            'Relative Date': test_df.iloc[:, 0],
            'Predict Target': predictions
        })
        filename = str(OUTPUT_DIR / f'submission_{model_name.lower()}.csv')
        submission.to_csv(filename, index=False)
        print(f"{model_name}预测结果已保存至 {filename}")
    
    # 保存集成预测（三个模型的平均）
    ensemble_predictions = np.mean([
        lstm_test_predictions_rescaled,
        transformer_test_predictions_rescaled,
        arima_test_predictions
    ], axis=0)
    
    ensemble_submission = pd.DataFrame({
        'Relative Date': test_df.iloc[:, 0],
        'Predict Target': ensemble_predictions
    })
    ensemble_submission.to_csv(str(OUTPUT_DIR / 'submission_ensemble.csv'), index=False)
    print("集成模型预测结果已保存至 submission_ensemble.csv")
    
    # ========== 10. 生成最终报告 ==========
    print("\n" + "="*80)
    print("10. 生成最终报告")
    print("="*80)
    
    report = f"""
{'='*80}
时间序列预测模型对比分析报告
{'='*80}

数据集信息:
------------
- 训练集大小: {len(train_df)}
- 验证集大小: {len(val_df)}
- 测试集大小: {len(test_df)}
- 序列长度: {seq_length}

模型性能对比:
------------
{comparison_df.to_string()}

最佳模型（按指标）:
------------------
"""
    
    for metric, (model, value) in best_model_by_metric.items():
        report += f"- {metric}: {model} ({value:.4f})\n"
    
    report += f"""

预测文件:
---------
- LSTM: submission_lstm.csv
- Transformer: submission_transformer.csv
- ARIMA: submission_arima.csv
- Ensemble (平均): submission_ensemble.csv

可视化文件:
----------
- LSTM训练历史: lstm_training_history.png
- LSTM预测结果: lstm_predictions.png
- LSTM残差分析: lstm_residuals.png

- Transformer训练历史: transformer_training_history.png
- Transformer预测结果: transformer_predictions.png
- Transformer残差分析: transformer_residuals.png

- ARIMA诊断图: arima_diagnostics_initial.png, arima_diagnostics_fitted.png
- ARIMA预测结果: arima_predictions.png
- ARIMA残差分析: arima_residuals.png

- 综合对比图: comprehensive_model_comparison.png

模型推荐:
--------
"""
    
    # 综合评分（越低越好）
    综合评分 = {}
    for model_name in results.keys():
        score = (
            comparison_df.loc[model_name, 'MSE'] / comparison_df['MSE'].max() +
            comparison_df.loc[model_name, 'MAE'] / comparison_df['MAE'].max() +
            comparison_df.loc[model_name, 'MAPE'] / comparison_df['MAPE'].max()
        ) / 3
        综合评分[model_name] = score
    
    最佳综合模型 = min(综合评分, key=综合评分.get)
    
    report += f"基于MSE、MAE、MAPE的综合评分，推荐使用: {最佳综合模型}\n"
    report += f"\n各模型综合评分（越低越好）:\n"
    for model_name, score in sorted(综合评分.items(), key=lambda x: x[1]):
        report += f"  - {model_name}: {score:.4f}\n"
    
    report += f"""
{'='*80}
分析完成！
{'='*80}
"""
    
    print(report)
    
    # 保存报告
    with open(str(OUTPUT_DIR / 'model_comparison_report.txt'), 'w', encoding='utf-8') as f:
        f.write(report)
    print("\n详细报告已保存至 model_comparison_report.txt")
    
    print("\n" + "="*80)
    print("所有任务完成！")
    print("="*80)

if __name__ == "__main__":
    main()